In [105]:
import requests
from datetime import datetime
import time

In [108]:
URL = 'http://eas-iot.ufsc.br/api/location'
# URL = 'http://localhost:5000/api/location'

In [109]:
def send_data(url,data):
    # Convert data to query parameters
    try:
        response = requests.get(url, params=data)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        return {"error": str(e)}

In [114]:
lat1 = -27.597775
lon1 = -48.425409

lat2 = -27.607125
lon2 = -48.414851

latspeed = lat2-lat1
lonspeed = lon1-lon2

points = [(lat1,lon1),(lat2,lon2)]

for i in range(100):
    latspeed -= 0.0001
    lat = points[-1][0] + latspeed
    lon = points[-1][1] + lonspeed
    points.append((lat,lon))

for i,point in enumerate(points):
    data = {
        "sender_id": '0',
        "timestamp": int(time.time()),
        "latitude": point[0],
        "longitude": point[1],
        "gps_module_id": '0',
        "battery_level": 100-i,
        "device_status": 'ok'
    }
    r = send_data(URL,data)

In [112]:
r

{'message': 'Dados recebidos e guardados com sucesso'}

In [50]:
import psycopg2

In [53]:
conn = psycopg2.connect(
    host="localhost",
    dbname="derivaiot",
    user="derivaiot",
    password="3vKRgfXP",
)

In [ ]:
cur = conn.cursor()

In [58]:
timestamp = datetime.fromtimestamp(data["timestamp"])
record = {
    "sender_id": data.get("sender_id"),
    "timestamp": timestamp,
    "latitude": data["latitude"],
    "longitude": data["longitude"],
    "gps_module_id": data["gps_module_id"],
    "battery_level": data.get("battery_level"),
    "device_status": data.get("device_status")
}
    
cur.execute(
    """INSERT INTO deriva_points (sender_id, timestamp, latitude, longitude, gps_module_id, battery_level, device_status)
        VALUES (%(sender_id)s, %(timestamp)s, %(latitude)s, %(longitude)s, %(gps_module_id)s, %(battery_level)s, %(device_status)s)""",
    record
)
conn.commit()